In [1]:
import os
import shutil

# 1. Tentukan path folder asal dan folder tujuan
# Sesuaikan jika nama foldernya berbeda di komputermu
source_dir = 'RespiScan_Train_Ready' 
mfcc_backup_dir = 'RespiScan_MFCC_Only'

# Folder utama dan sub-kelas yang biasa digunakan
phases = ['TRAIN', 'VAL', 'TEST']
classes = ['TB', 'Healthy']

pindah_count = 0
sisa_count = 0

print("🚀 Memulai proses pemisahan MFCC dan Scalogram...")

# 2. Iterasi melalui setiap folder dan subfolder
for phase in phases:
    for cls in classes:
        # Path spesifik, misal: 'RespiScan_Train_Ready/TRAIN/TB'
        current_dir = os.path.join(source_dir, phase, cls)
        backup_dir = os.path.join(mfcc_backup_dir, phase, cls)
        
        # Buat folder backup jika belum ada
        if not os.path.exists(backup_dir):
            os.makedirs(backup_dir)
            
        # Cek apakah folder asal benar-benar ada
        if os.path.exists(current_dir):
            files = os.listdir(current_dir)
            
            for file in files:
                # Cek apakah nama file mengandung kata 'mfcc' (huruf besar/kecil tidak masalah)
                if 'mfcc' in file.lower():
                    # Path lengkap untuk file asal dan tujuan
                    src_file = os.path.join(current_dir, file)
                    dst_file = os.path.join(backup_dir, file)
                    
                    # Pindahkan file
                    shutil.move(src_file, dst_file)
                    pindah_count += 1
                else:
                    # Jika tidak ada kata 'mfcc', asumsikan ini adalah Scalogram
                    sisa_count += 1

print("-" * 40)
print(f"✅ Pemisahan Selesai!")
print(f"📁 Total file MFCC yang dipindah: {pindah_count} gambar.")
print(f"🖼️ Total file Scalogram yang tersisa: {sisa_count} gambar.")
print(f"Folder '{source_dir}' sekarang sudah BERSIH dan siap untuk training!")

🚀 Memulai proses pemisahan MFCC dan Scalogram...
----------------------------------------
✅ Pemisahan Selesai!
📁 Total file MFCC yang dipindah: 2326 gambar.
🖼️ Total file Scalogram yang tersisa: 3322 gambar.
Folder 'RespiScan_Train_Ready' sekarang sudah BERSIH dan siap untuk training!


In [1]:
#model perbaikan
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import f1_score
import time
import copy
import os

# ==========================================
# 0. ALGORITMA FOCAL LOSS (SENJATA BARU)
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha # Bobot kelas (opsional)
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean': return focal_loss.mean()
        elif self.reduction == 'sum': return focal_loss.sum()
        return focal_loss

# ==========================================
# 1. SETUP HARDWARE
# ==========================================
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"🚀 Menjalankan training di perangkat: {DEVICE}")

# ==========================================
# 2. SETUP DATASET & AUGMENTASI BARU
# ==========================================
DATA_DIR = 'RespiScan_Train_Ready' # Pastikan folder ini berisi murni Scalogram

# Transformasi Training Diperkuat (Solusi 3)
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5), 
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.05)), # Geser (Time/Freq Shifting)
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_DIR, x.upper()), data_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=32, shuffle=True, num_workers=0) for x in ['train', 'val']}

# ==========================================
# 3. SETUP MOBILENETV2 & FOCAL LOSS
# ==========================================
model = models.mobilenet_v2(weights='DEFAULT')

for param in model.parameters(): param.requires_grad = True

num_ftrs = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 2)
)
model = model.to(DEVICE)

optimizer = optim.Adam([
    {'params': model.features[14:].parameters(), 'lr': 1e-5}, 
    {'params': model.classifier.parameters(), 'lr': 1e-4}      
], lr=1e-5) 

# Gunakan Focal Loss dengan Bobot 2:1 (Solusi 1)
weights = torch.tensor([2.0, 1.0]).to(DEVICE)
criterion = FocalLoss(alpha=weights, gamma=2.0)
print("🎯 Model menggunakan FOCAL LOSS untuk mendeteksi kasus sulit!")

# ==========================================
# 4. FUNGSI TRAINING
# ==========================================
def train_model_v5(model, criterion, optimizer, num_epochs=35, patience=8):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_f1 = 0.0
    early_stop_counter = 0
    
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train': model.train()
            else: model.eval()

            running_loss, running_corrects = 0.0, 0
            all_preds, all_labels = [], []

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])
            epoch_f1 = f1_score(all_labels, all_preds, zero_division=0)

            print(f'{phase.upper()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} F1: {epoch_f1:.4f}')

            if phase == 'val':
                scheduler.step(epoch_f1)
                if epoch_f1 > best_f1:
                    best_f1 = epoch_f1
                    best_model_wts = copy.deepcopy(model.state_dict())
                    torch.save(model.state_dict(), 'best_RespiScan_FocalLoss.pth')
                    early_stop_counter = 0
                    print("⭐️ Model Terbaik (Focal Loss) Disimpan!")
                else:
                    early_stop_counter += 1

        if early_stop_counter >= patience:
            print(f"🛑 Early Stopping di epoch {epoch+1}.")
            break

    print(f'\n🏆 Best Validation F1-Score: {best_f1:.4f}')
    model.load_state_dict(best_model_wts)
    return model, history

# ==========================================
# 5. EKSEKUSI
# ==========================================
model_trained, history = train_model_v5(model, criterion, optimizer, num_epochs=35)

🚀 Menjalankan training di perangkat: cuda:0
🎯 Model menggunakan FOCAL LOSS untuk mendeteksi kasus sulit!

Epoch 1/35
----------
TRAIN Loss: 0.2805 Acc: 0.5919 F1: 0.1371
VAL Loss: 0.3596 Acc: 0.5291 F1: 0.1415
⭐️ Model Terbaik (Focal Loss) Disimpan!

Epoch 2/35
----------
TRAIN Loss: 0.2320 Acc: 0.6308 F1: 0.2651
VAL Loss: 0.3783 Acc: 0.5591 F1: 0.3842
⭐️ Model Terbaik (Focal Loss) Disimpan!

Epoch 3/35
----------
TRAIN Loss: 0.2189 Acc: 0.6770 F1: 0.4223
VAL Loss: 0.3905 Acc: 0.5785 F1: 0.4947
⭐️ Model Terbaik (Focal Loss) Disimpan!

Epoch 4/35
----------
TRAIN Loss: 0.2045 Acc: 0.7155 F1: 0.5307
VAL Loss: 0.3647 Acc: 0.5714 F1: 0.3879

Epoch 5/35
----------
TRAIN Loss: 0.2039 Acc: 0.7190 F1: 0.5397
VAL Loss: 0.3906 Acc: 0.5891 F1: 0.5196
⭐️ Model Terbaik (Focal Loss) Disimpan!

Epoch 6/35
----------
TRAIN Loss: 0.1907 Acc: 0.7367 F1: 0.5837
VAL Loss: 0.4048 Acc: 0.5785 F1: 0.5172

Epoch 7/35
----------
TRAIN Loss: 0.1876 Acc: 0.7648 F1: 0.6449
VAL Loss: 0.3908 Acc: 0.5855 F1: 0.4968
